# 04 模型关注区域与切片权重

这本 Notebook 从两个角度查看 ResNet 的预测：用 Grad-CAM 显示每层影像中与阳性输出相关的区域，用注意力权重观察模型如何汇总四层信息。分析同时覆盖预测正确和预测错误的病例。

需要先完成 02 的开发队列五折训练。每位患者使用其所在留出折对应的模型来生成热图，便于与该患者的折外预测对应。

下面先读取实验配置，再定义模型和可视化函数。


In [ ]:
from pathlib import Path

HERE = Path.cwd().resolve()
CANDIDATES = [HERE, HERE / "1117", HERE.parent / "1117"]
NB_ROOT = next((p for p in CANDIDATES if (p / "config.json").is_file()), None)
if NB_ROOT is None:
    raise FileNotFoundError("请从保存config.json的Notebook目录打开；不需要外层code目录。")


## 实验配置与文件记录

这一组函数负责管理实验文件。`load_config` 读取 `config.json`，`paths` 整理缓存、模型和报告的保存位置；后面的单元通过 `cfg` 读取参数，通过 `p` 查找文件。

`read_cohort` 读取患者清单，`development` 从中取出开发队列。保存结果时，`record` 会同时记录配置和代码版本，便于回查某次结果是怎样得到的。`fresh` 用来检查目标文件是否已存在，重新开展一组实验时可以换一个 `run_id`。


In [ ]:
"""Configuration, provenance and stage boundaries; standard library only."""
from __future__ import annotations

import hashlib
import json
import re
from pathlib import Path

SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]


def load_config(path: str | Path) -> dict:
    path = Path(path).resolve()
    cfg = json.loads(path.read_text(encoding="utf-8"))
    settings = path.parent / "experiment_settings.json"
    if settings.is_file():
        overlay = json.loads(settings.read_text(encoding="utf-8"))
        allowed = {"annotation_scope", "optimized", "radiomics", "segmentation", "report_version"}
        if set(overlay) - allowed:
            raise ValueError("experiment_settings.json 只放实验参数，机器路径仍在 config.json。")
        for key, value in overlay.items():
            if isinstance(value, dict):
                cfg[key] = {**cfg.get(key, {}), **value}
            else:
                cfg[key] = value
    if not re.fullmatch(r"[A-Za-z0-9_-]+", cfg["run_id"]):
        raise ValueError("run_id 只使用字母、数字、下划线或短横线。")
    cfg["_root"] = path.parent
    for key in ("excel_path", "patients_root", "existing_cache", "outputs_root"):
        if cfg.get(key):
            p = Path(cfg[key]).expanduser()
            cfg[key] = p if p.is_absolute() else path.parent / p
    cfg["_run"] = cfg["outputs_root"] / cfg["run_id"]
    if cfg.get("data_access") == "classification_cache_only":
        verify_classification_bundle(cfg)
    return cfg




def paths(cfg: dict) -> dict[str, Path]:
    root = cfg["_run"]
    return {
        "run": root, "data": root / "data", "cache": Path(cfg.get("existing_cache") or root / "data" / "resnet50_manual_cache.npz"),
        "adapter": root / "data" / "clinical_adapter.xlsx", "cohort": root / "data" / "cohort.csv",
        "temporal": root / "temporal", "full_cv": root / "full_cv", "single": root / "single_sequence",
        "radiomics": root / "radiomics" / experiment_name(cfg, "radiomics", "selected2d_v1"),
        "explain": root / "explain", "segmentation": root / "segmentation" / experiment_name(cfg, "segmentation", "selected2d_v1"),
        "optimized": root / "optimized" / experiment_name(cfg, "optimized", "resnet_v1"),
        "reports": root / "reports" / safe_name(cfg.get("report_version", "comparison_v2")),
    }




def dump_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")




def fresh(*targets: Path) -> None:
    exists = [str(p) for p in targets if p.exists()]
    if exists:
        raise FileExistsError("产物已存在，请加载已有结果，或更换 run_id：" + ", ".join(exists))




def digest(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()




def source_snapshot(cfg: dict) -> dict:
    """Hash the literal definition cells, independently of outputs and execution counts."""
    result = {}
    for path in sorted(cfg["_root"].glob("*.ipynb")):
        notebook = json.loads(path.read_text(encoding="utf-8"))
        definitions = ["".join(cell["source"]) for cell in notebook["cells"]
                       if cell["cell_type"] == "code" and cell.get("metadata", {}).get("embedded_definition")]
        result[path.name] = hashlib.sha256("\n\n".join(definitions).encode("utf-8")).hexdigest()
    return result




def record(cfg: dict, stage: str, extra: dict | None = None) -> None:
    obj = {"stage": stage, "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
           "notebook_definitions_sha256": source_snapshot(cfg),
           "protocol": "resnet_baseline_and_optimized_selected_slices", **(extra or {})}
    dump_json(paths(cfg)["run"] / "provenance" / f"{stage}.json", obj)




def read_cohort(cfg: dict):
    import pandas as pd
    p = paths(cfg)["cohort"]
    if not p.exists():
        raise FileNotFoundError("先运行00的数据准备与病例清单单元。")
    table = pd.read_csv(p, dtype={"mr": str})
    if not table["mr"].is_unique or not set(table["split"]) <= {"development", "validation"}:
        raise ValueError("病例清单编号重复或队列标记无效。")
    return table




def development(cfg: dict):
    return read_cohort(cfg).query("split == 'development'").reset_index(drop=True)




def require_input(cfg: dict, key: str) -> Path:
    value = cfg.get(key)
    if not value or not Path(value).exists():
        raise FileNotFoundError(f"请在 config.json 配置有效的 {key}。")
    return Path(value)




def verify_classification_bundle(cfg: dict) -> None:
    """Verify portable data without accessing the original image directories."""
    root = cfg["_root"].resolve()
    manifest_path = root / "classification_bundle.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("分类数据包缺少 classification_bundle.json，请上传完整目录。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("kind") != "classification_cache_only" or manifest["run_id"] != cfg["run_id"]:
        raise ValueError("分类数据包与当前 run_id 不一致。")
    manifest["artifacts"] = {portable_name(k): v for k, v in manifest["artifacts"].items()}
    for key in ("cache", "cohort"):
        manifest[key] = portable_name(manifest[key])
    p = paths(cfg)
    for key in ("cache", "cohort"):
        expected = (root / manifest[key]).resolve()
        expected.relative_to(root)
        if p[key].resolve() != expected or manifest[key] not in manifest["artifacts"]:
            raise ValueError(f"分类数据包的 {key} 路径或清单被改变。")
    for name, expected_hash in manifest["artifacts"].items():
        file = (root / name).resolve()
        file.relative_to(root)
        if not file.is_file() or digest(file) != expected_hash:
            raise ValueError(f"分类数据包文件缺失或校验失败：{name}")


    # Notebook hashes are provenance, not data identity. Model contracts are checked separately.


def require_native_images(cfg: dict) -> None:
    if cfg.get("data_access") == "classification_cache_only":
        raise ValueError("本操作需要原分辨率影像，请在数据电脑执行；导入结果和生成报告无需原影像。")




def safe_name(value):
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError("实验名称只使用字母、数字、下划线或短横线。")
    return value




def experiment_name(cfg, stage, default):
    return safe_name(cfg.get(stage, {}).get("experiment", default))




def portable_name(value):
    from pathlib import PurePosixPath
    name = str(value).replace("\\", "/")
    path = PurePosixPath(name)
    if path.is_absolute() or not path.parts or any(p in ("..", ".") or ":" in p for p in path.parts):
        raise ValueError(f"数据包必须使用包内相对路径：{value}")
    return path.as_posix()




def contained_file(root, name):
    root = Path(root).resolve()
    target = (root / portable_name(name)).resolve()
    target.relative_to(root)
    return target




def cohort_identity(cfg):
    """Patient identity independent of local directories, CSV formatting and row order."""
    import pandas as pd
    table = read_cohort(cfg)
    columns = ["mr", "label", "split", "development_fold", "max_layer", "moca"] + CLINICAL
    if not set(columns) <= set(table):
        raise ValueError("病例清单缺少患者、结局、分组、取层或临床列。")
    rows = []
    for _, row in table.sort_values("mr").iterrows():
        values = {}
        for key in columns:
            value = row[key]
            values[key] = (None if pd.isna(value) else str(value) if key in ("mr", "split") else format(float(value), ".12g"))
        rows.append(values)
    payload = {"schema": 1, "patients": rows}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()




def definition_contract(cfg, prefix, names):
    import ast
    files = list(cfg["_root"].glob(prefix + "_*.ipynb"))
    if len(files) != 1:
        raise ValueError(f"找不到唯一的 {prefix} Notebook。")
    notebook = json.loads(files[0].read_text(encoding="utf-8"))
    definitions = {}
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        for node in ast.parse("".join(cell["source"])).body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names:
                if node.name in definitions:
                    raise ValueError(f"重复定义：{node.name}")
                definitions[node.name] = hashlib.sha256(ast.dump(node, include_attributes=False).encode()).hexdigest()
    if set(definitions) != set(names):
        raise ValueError("模型或处理流程定义不齐全。")
    return definitions




def verify_transfer(cfg, directory, kind=None):
    directory = Path(directory).resolve()
    manifest = json.loads((directory / "transfer_manifest.json").read_text(encoding="utf-8"))
    if manifest.get("schema") != 1 or manifest.get("kind") not in ("radiomics_features", "radiomics_complete", "segmentation_results", "segmentation_weights"):
        raise ValueError("不支持的数据包类型或版本。")
    if kind and manifest["kind"] != kind:
        raise ValueError("数据包类型不匹配。")
    if manifest["cohort_identity"] != cohort_identity(cfg):
        raise ValueError("两台机器的患者、标签、临床数据、取层或开发折不一致。")
    if manifest["native_source_identity"] != native_source_identity(cfg):
        raise ValueError("数据包与分类缓存记录的原始影像或人工标注不是同一版本。")
    if manifest["annotation_scope"] != "selected_slices" or cfg["annotation_scope"] != "selected_slices":
        raise ValueError("当前流程只接受四层标注协议。")
    for name, checksum in manifest["artifacts"].items():
        file = contained_file(directory, name)
        if not file.is_file() or digest(file) != checksum:
            raise ValueError(f"数据包文件缺失或传输后发生变化：{name}")
    return manifest




def export_transfer(cfg, source, filenames, destination, kind, protocol):
    import shutil
    source, destination = Path(source).resolve(), Path(destination).resolve()
    fresh(destination)
    mapping = {}
    for name in filenames:
        name = portable_name(name)
        file = contained_file(source, name)
        if not file.is_file():
            raise FileNotFoundError(file)
        mapping[name] = digest(file)
    destination.mkdir(parents=True)
    for name in mapping:
        target = contained_file(destination, name)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), target)
    dump_json(destination / "transfer_manifest.json", {
        "schema": 1, "kind": kind, "cohort_identity": cohort_identity(cfg), "native_source_identity": native_source_identity(cfg),
        "annotation_scope": cfg["annotation_scope"], "protocol": protocol,
        "artifacts": mapping, "source_run_id": cfg["run_id"],
        "notebook_definitions_sha256": source_snapshot(cfg)})
    verify_transfer(cfg, destination, kind)
    return destination




def import_transfer(cfg, source, stage):
    import shutil
    source = Path(source).resolve()
    manifest = verify_transfer(cfg, source)
    allowed = {"radiomics": ("radiomics_features", "radiomics_complete"),
               "segmentation": ("segmentation_results",), "segmentation_weights": ("segmentation_weights",)}
    if stage not in allowed or manifest["kind"] not in allowed[stage]:
        raise ValueError("数据包与接收阶段不匹配。")
    expected_protocol = "selected_four_slices_2d_pixels_v1"
    if manifest["protocol"] != expected_protocol:
        raise ValueError("数据包的二维取层/像素空间处理协议不匹配。")
    target = paths(cfg)[stage] if stage != "segmentation_weights" else paths(cfg)["segmentation"].with_name(paths(cfg)["segmentation"].name + "_imported_model")
    fresh(target)
    target.mkdir(parents=True)
    for name in manifest["artifacts"]:
        dest = contained_file(target, name)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(contained_file(source, name), dest)
    shutil.copy2(source / "transfer_manifest.json", target / "transfer_manifest.json")
    verify_transfer(cfg, target)
    return target




def native_source_identity(cfg):
    audit_file = paths(cfg)["data"] / "volume_audit_details.json"
    audit = json.loads(audit_file.read_text(encoding="utf-8"))
    patients = read_cohort(cfg).mr.tolist()
    files = IMAGE_FILES + ["Untitled.nii.gz"]
    selected = {mr: {name: audit["patients"][mr]["files"][name]["sha256"] for name in files} for mr in patients}
    return hashlib.sha256(json.dumps(selected, sort_keys=True).encode()).hexdigest()




In [ ]:
cfg = load_config(NB_ROOT / "config.json")
p = paths(cfg)
print("实验：", cfg["run_id"])
print("结果目录：", p["run"])


## 表格与数组处理

NumPy 用于处理影像数组和预测概率，pandas 用于读取患者清单、合并临床信息及整理结果表。JSON 用于保存参数和指标；正则表达式用于检查编号等文本字段。


In [ ]:
from __future__ import annotations
import json
import re
import numpy as np
import pandas as pd


## 模型计算与图像处理工具

这里加载 PyTorch 和 ResNet50，供后续恢复训练好的网络。计算设备与模型加载位置保持一致；绘图和结果表所需的工具会在相应函数中使用。


In [ ]:
from __future__ import annotations
import copy
import random
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

SEED = 20260922
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SINGLE_SEQUENCES = {"t1": 0, "t2": 1, "dwi": 2}
CACHE = OUT_DIR = TEMPORAL_OUT = TEMPORAL_SPLIT = None
PREP_BASE = PREP_OUT = ADAPTER_XLSX = None


## 连接输入与输出目录

`configure_baseline` 把配置中确定的文件位置传给后续函数，包括影像缓存、临床表、时间分组表和模型输出目录。后面调用数据处理或模型函数时，就会使用同一组路径。


In [ ]:
from __future__ import annotations

def configure_baseline(cache, full_cv, temporal, adapter, patients_root=None):
    """Configure only input/output locations; do not alter the training recipe."""
    global CACHE, OUT_DIR, TEMPORAL_OUT, TEMPORAL_SPLIT
    global PREP_BASE, PREP_OUT, ADAPTER_XLSX
    CACHE, OUT_DIR, TEMPORAL_OUT = Path(cache), Path(full_cv), Path(temporal)
    TEMPORAL_SPLIT, ADAPTER_XLSX = TEMPORAL_OUT / "temporal_split.csv", Path(adapter)
    PREP_BASE, PREP_OUT = Path(patients_root) if patients_root else None, CACHE


## 恢复分类模型的结构

`MaskEncoder` 处理三类病灶掩模，`DualBranchModel` 把病灶特征与 ResNet50 的影像特征拼接，再通过四层注意力得到患者级预测。

这里需要定义与训练时一致的网络结构，才能正确载入 02 保存的参数。后面的热图分析会读取 ResNet 最后一组卷积的输出，以及四层注意力的分数。


In [ ]:
from __future__ import annotations


# 直接复制：code/train_resnet50_manual_cv.py :: MaskEncoder
class MaskEncoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x.float())


# 直接复制：code/train_resnet50_manual_cv.py :: DualBranchModel
class DualBranchModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.mask_features = MaskEncoder()
        self.attention = nn.Sequential(
            nn.Linear(2304, 256),
            nn.Tanh(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )
        self.head = nn.Sequential(
            nn.LayerNorm(2304),
            nn.Dropout(0.4),
            nn.Linear(2304, 1),
        )

    def forward(self, images: torch.Tensor, masks: torch.Tensor) -> torch.Tensor:
        B, S, C, H, W = images.shape
        im = images.view(B * S, C, H, W)
        image_feat = self.image_features(im)
        image_feat = torch.flatten(image_feat, 1)
        image_feat = image_feat.view(B, S, -1)

        mask = masks.view(B * S, 3, H, W)
        mask_feat = self.mask_features(mask)
        mask_feat = mask_feat.view(B, S, -1)

        feat = torch.cat([image_feat, mask_feat], dim=-1)
        attn = self.attention(feat).squeeze(-1)
        attn = torch.softmax(attn, dim=1)
        pooled = (feat * attn.unsqueeze(-1)).sum(dim=1)
        return self.head(pooled).squeeze(-1)


## 加载患者对应的折模型

`load_original_model` 根据传入的权重路径构建网络，严格匹配参数名称和形状，再切换到评估状态。解释函数会为每位患者传入 `dev_fold_*.pt`，选择该患者作为留出病例时的模型。


In [ ]:
from __future__ import annotations

def configure(cfg):
    p = paths(cfg)
    configure_baseline(p["cache"], p["full_cv"], p["temporal"], p["adapter"], cfg.get("patients_root"))
    return p







def load_original_model(cfg, checkpoint=None):
    import torch
    configure(cfg)
    model = DualBranchModel().to(DEVICE)
    checkpoint = checkpoint or paths(cfg)["temporal"] / "final_development_model.pt"
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True), strict=True)
    model.eval()
    return model







In [ ]:
from __future__ import annotations

def baseline_contract(cfg):
    return definition_contract(cfg, "02", list(ORIGINAL_INFERENCE_CONTRACT))






ORIGINAL_INFERENCE_CONTRACT = {'MaskEncoder': 'b0a35f2673a19f24a5f492a757a65894b572a9819996d2aabb9572cc89b2de04', 'DualBranchModel': '75a158f454de456d8b6e452309c35eb101124771da83567a475d8af2e386bc95', 'predict_proba': '94646b415f42d60d4a16416a05f26afed9656dd0fba15b09323c50f226898663'}


## 恢复优化模型

这组定义与02的优化网络一致，载入时核对权重、参数、队列与缓存。推理直接从已保存权重恢复，不需要下载ImageNet预训练参数。


In [ ]:
from __future__ import annotations

class GatedSliceAttention(nn.Module):
    def __init__(self, features, hidden):
        super().__init__()
        self.value = nn.Linear(features, hidden)
        self.gate = nn.Linear(features, hidden)
        self.score = nn.Linear(hidden, 1)

    def forward(self, features):
        return self.score(torch.tanh(self.value(features)) * torch.sigmoid(self.gate(features)))






class CompactMaskEncoder(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, 128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, dim), nn.LayerNorm(dim), nn.GELU(), nn.Dropout(.3))

    def forward(self, masks):
        return self.features(masks.float())






class OptimizedResNet(nn.Module):
    def __init__(self, settings, pretrained=True):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.image_projection = nn.Sequential(nn.Linear(2048, settings["image_dim"]),
            nn.LayerNorm(settings["image_dim"]), nn.GELU(), nn.Dropout(.3))
        self.mask_features = CompactMaskEncoder(settings["mask_dim"])
        features = settings["image_dim"] + settings["mask_dim"]
        self.attention = GatedSliceAttention(features, settings["attention_dim"])
        self.head = nn.Sequential(nn.LayerNorm(features), nn.Dropout(.4), nn.Linear(features, 1))

    def train(self, mode=True):
        super().train(mode)
        # Keep pretrained running means/variances stable with small patient batches.
        for module in self.image_features.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()
        return self

    def set_training_stage(self, fine_tune):
        for parameter in self.image_features.parameters():
            parameter.requires_grad_(False)
        if fine_tune:
            for parameter in self.image_features[7].parameters():
                parameter.requires_grad_(True)
        for module in self.image_features.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                for parameter in module.parameters():
                    parameter.requires_grad_(False)

    def forward(self, images, masks):
        b, slices, channels, h, w = images.shape
        image = self.image_features(images.reshape(b * slices, channels, h, w)).flatten(1)
        image = self.image_projection(image).reshape(b, slices, -1)
        mask = self.mask_features(masks.reshape(b * slices, 3, h, w)).reshape(b, slices, -1)
        features = torch.cat([image, mask], dim=-1)
        attention = torch.softmax(self.attention(features).squeeze(-1), dim=1)
        return self.head((features * attention.unsqueeze(-1)).sum(dim=1)).squeeze(-1)






def optimized_contract(cfg):
    return definition_contract(cfg, "02", ["GatedSliceAttention", "CompactMaskEncoder", "OptimizedResNet",
        "optimized_predict", "fit_optimized_epochs", "run_optimized_training", "optimized_data",
        "seed_everything", "augment_batch", "make_criterion"])






def verify_optimized(cfg):
    p = paths(cfg)
    lock = json.loads((p["optimized"] / "frozen_artifacts.json").read_text(encoding="utf-8"))
    if lock["cohort_identity"] != cohort_identity(cfg) or lock["cache_sha256"] != digest(p["cache"]) or lock["code_contract"] != optimized_contract(cfg):
        raise ValueError("优化模型与当前患者、缓存或实现不匹配。")
    for name, checksum in lock["artifacts"].items():
        if digest(p["optimized"] / name) != checksum:
            raise ValueError(f"优化模型文件发生变化：{name}")
    settings = json.loads((p["optimized"] / "training_config.json").read_text(encoding="utf-8"))["settings"]
    if settings != cfg["optimized"]:
        raise ValueError("优化参数与训练时不一致；加载已有模型时使用原参数，调参请更换 experiment。")
    return settings






def load_optimized_model(cfg, checkpoint=None):
    settings = verify_optimized(cfg)
    model = OptimizedResNet(settings, pretrained=False).to(DEVICE)
    checkpoint = checkpoint or paths(cfg)["optimized"] / "final_model.pt"
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True), strict=True)
    model.eval()
    return model






## 选择病例并生成 Grad-CAM

`choose_development_cases` 根据开发折外概率和已保存的阈值，把患者分为真阳性（TP）、真阴性（TN）、假阳性（FP）和假阴性（FN），按 MR 排序后各取一例，形成固定的初始展示规则。

`grad_cam` 读取 ResNet 最后一组卷积特征，计算患者阳性输出分数对这些特征的梯度。梯度用于加权各特征通道，得到每层热图，再放大到 224 × 224。函数还会读取四层注意力分数，并转成总和为 1 的权重。

`explain_development_cases` 负责载入患者和折模型、调用热图函数，并保存三序列图像、病灶轮廓、热图及权重。三个 MRI 序列在网络中共同形成特征，因此每层得到一张联合输入的热图。


In [ ]:
"""New explanation utilities; original model forward and weights remain untouched."""
from __future__ import annotations

import json
import numpy as np
import pandas as pd






def choose_development_cases(cfg, model_kind="baseline"):
    p, dev = paths(cfg), development(cfg)
    folder = p["optimized"] if model_kind == "optimized" else p["temporal"]
    oof = pd.read_csv(folder / "development_oof.csv", dtype={"mr": str}).rename(columns={"probability": "dev_oof_prob"})
    table = dev.merge(oof[["mr", "dev_oof_prob"]], on="mr", validate="one_to_one")
    filename, key = ("development_metrics.json", "threshold") if model_kind == "optimized" else ("development_cv_metrics.json", "youden_threshold")
    threshold = json.loads((folder / filename).read_text(encoding="utf-8"))[key]
    predicted = table.dev_oof_prob >= threshold
    table["category"] = np.select([(table.label == 1) & predicted, (table.label == 0) & ~predicted,
                                  (table.label == 0) & predicted], ["TP", "TN", "FP"], default="FN")
    # Fixed MR ordering for reproducible example selection.
    return table.sort_values("mr").groupby("category", sort=True).head(1)







def grad_cam(model, images, masks, target_class=1):
    import torch
    import torch.nn.functional as F
    if target_class not in (0, 1) or len(images) != 1:
        raise ValueError("当前解释入口每次一个患者，target_class为0或1。")
    captured = {}

    def feature_hook(_module, _inputs, output):
        captured["features"] = output
        output.retain_grad()

    def attention_hook(_module, _inputs, output):
        captured["attention"] = output

    # Original ResNet sequential: conv1,bn,relu,maxpool,layer1..layer4,avgpool.
    feature_handle = model.image_features[7].register_forward_hook(feature_hook)
    attention_handle = model.attention.register_forward_hook(attention_hook)
    model.eval()
    try:
        with torch.enable_grad():
            model.zero_grad(set_to_none=True)
            logits = model(images.requires_grad_(True), masks)
            (logits if target_class == 1 else -logits).sum().backward()
            feature = captured["features"]
            weights = feature.grad.mean(dim=(2, 3), keepdim=True)
            maps = torch.relu((weights * feature).sum(dim=1, keepdim=True))
            maps = F.interpolate(maps, size=images.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
            maximum = maps.amax(dim=(1, 2), keepdim=True)
            maps = maps / maximum.clamp_min(1e-8)
            attention = torch.softmax(captured["attention"].squeeze(-1), dim=1)[0]
            return {"cam": maps.detach().cpu().numpy(), "attention": attention.detach().cpu().numpy(),
                    "probability": float(torch.sigmoid(logits)[0].detach().cpu())}
    finally:
        feature_handle.remove()
        attention_handle.remove()
        model.zero_grad(set_to_none=True)







def explain_development_cases(cfg, selected_mrs=None, model_kind="baseline"):
    import matplotlib.pyplot as plt
    import torch
    p, dev = paths(cfg), development(cfg)
    if model_kind not in ("baseline", "optimized"):
        raise ValueError("model_kind只能是baseline或optimized。")
    if model_kind == "optimized":
        p["explain"] = p["optimized"] / "explain"
    if selected_mrs:
        if not set(selected_mrs) <= set(dev.mr):
            raise ValueError("本入口只解释开发队列的折外患者。")
        selected = dev.loc[dev.mr.isin(selected_mrs)]
    else:
        selected = choose_development_cases(cfg, model_kind)
    fresh(p["explain"] / "cases.csv")
    p["explain"].mkdir(parents=True, exist_ok=True)
    configure(cfg)
    rows = []
    with np.load(p["cache"], allow_pickle=False) as cache:
        for _, patient in selected.iterrows():
            fold = int(patient.development_fold)
            checkpoint = p["temporal"] / f"dev_fold_{fold}.pt"
            if model_kind == "optimized":
                checkpoint = p["optimized"] / f"fold_{fold}.pt"
                model = load_optimized_model(cfg, checkpoint)
            else:
                model = load_original_model(cfg, checkpoint)
            idx = int(patient.cache_index)
            images = cache["images"][idx].astype(np.float32)
            masks = cache["masks"][idx].astype(np.uint8)
            explanation = grad_cam(model, torch.from_numpy(images[None]).to(DEVICE),
                                   torch.from_numpy(masks[None]).to(DEVICE))
            np.savez_compressed(p["explain"] / f"{patient.mr}_cam.npz", **explanation)
            fig, axes = plt.subplots(4, 4, figsize=(12, 12))
            for s in range(4):
                for c, title in enumerate(("T1-FLAIR", "T2", "DWI")):
                    axes[c, s].imshow(images[s, c], cmap="gray", vmin=0, vmax=1)
                    axes[c, s].set_title(f"{title} | slice {s + 1}")
                axes[3, s].imshow(images[s, 0], cmap="gray", vmin=0, vmax=1)
                axes[3, s].imshow(explanation["cam"][s], cmap="jet", alpha=.4, vmin=0, vmax=1)
                for label, color in enumerate(("cyan", "lime", "magenta")):
                    if masks[s, label].any() and not masks[s, label].all():
                        axes[3, s].contour(masks[s, label], levels=[.5], colors=[color], linewidths=.7)
                axes[3, s].set_title(f"CAM | attention={explanation['attention'][s]:.3f}")
            for ax in axes.flat:
                ax.axis("off")
            fig.suptitle(f"{patient.mr} | OOF fold={fold} | y={patient.label} | p={explanation['probability']:.3f}")
            fig.tight_layout()
            fig.savefig(p["explain"] / f"{patient.mr}.png", dpi=150)
            plt.close(fig)
            rows.append({"mr": patient.mr, "label": int(patient.label), "fold": fold,
                         "probability": explanation["probability"], **{f"slice_{i+1}_weight": float(w) for i, w in enumerate(explanation["attention"])}})
            del model
    table = pd.DataFrame(rows)
    table.to_csv(p["explain"] / "cases.csv", index=False)
    dump_json(p["explain"] / "method.json", {"target": "patient_positive_logit", "layer": "image_features[7]",
        "case_selection": "development fold model; first MR per TP/TN/FP/FN unless explicit list",
        "model_kind": model_kind, "limitation": "原基线曾用外折早停；优化版在内层选轮数。热图和切片权重不是分割结果或因果证明。"})
    return table







## 1. 查看准备展示的病例

表中列出各类病例的 MR 编号、真实标签、预测概率和开发折号。先确认是否覆盖了正确和错误预测；某一类没有患者时，该类不会出现在表中。


In [ ]:
if (p["temporal"] / "development_oof.csv").exists():
    display(choose_development_cases(cfg)[["mr", "label", "category", "development_fold", "dev_oof_prob"]])


## 2. 生成病例图和切片权重

`SELECTED_MRS` 留空时使用上面的默认病例；也可以填写开发患者的 MR 编号，查看指定病例。

每位患者输出一张四列图：每列对应一个切片，前三行依次显示 T1-FLAIR、T2、DWI，第四行在 T1-FLAIR 上叠加热图和病灶轮廓。图标题显示预测概率，切片标题显示注意力权重；数值结果另存为 NPZ 和 CSV。


In [ ]:
RUN_EXPLANATIONS = False
MODEL_KIND = "optimized"  # baseline 或 optimized
SELECTED_MRS = None
if RUN_EXPLANATIONS:
    display(explain_development_cases(cfg, selected_mrs=SELECTED_MRS, model_kind=MODEL_KIND))


## 3. 查看热图和权重表

查看模型关注区域与病灶轮廓的位置关系，并比较四层的注意力分配。对误判病例，可以结合影像质量、病灶位置和临床信息记录可能的原因，后续再设计实验验证。


In [ ]:
import pandas as pd
from IPython.display import Image, display
explain_folder = p["optimized"] / "explain" if MODEL_KIND == "optimized" else p["explain"]
if (explain_folder / "cases.csv").exists():
    cases = pd.read_csv(explain_folder / "cases.csv")
    display(cases)
    for mr in cases.mr:
        display(Image(filename=str(explain_folder / f"{mr}.png")))


## 热图与权重怎样解读

注意力权重反映模型在汇总四层特征时的相对分配；Grad-CAM 显示与当前输出相关的图像区域。热图在每层内部归一化，因此适合看空间分布，不能只凭两层颜色深浅比较贡献大小。

病灶轮廓可用于辅助观察，但热图与病灶重合并不能直接证明因果关系，也不能用来代替分割评价。各折模型曾利用留出折选择最佳轮数，这也是解释开发病例时需要记录的条件。

后续可通过遮挡实验、模型参数随机化或重复训练，进一步检查解释是否稳定。方法背景见 [Grad-CAM 论文](https://arxiv.org/abs/1610.02391) 和 [显著图检查研究](https://arxiv.org/abs/1810.03292)。


优化版热图保存到 `optimized/<experiment>/explain/`。原模型热图仍保存在原解释目录；图中的切片权重表示该模型的相对关注程度，不等同于病灶分割或因果依据。
